In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import *

In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("DDMFA")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/30 09:03:12 WARN Utils: Your hostname, NOMAAN-ANV15, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/30 09:03:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/nomaan/projects/Drone_Delivery_Monitoring_and_Failure_Analysis_Pipeline/ddmfa/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/nomaan/.ivy2.5.2/cache
The jars for the packages stored in: /home/nomaan/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-423488e9-12f1-49e0-a351-2528213c5218;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.3.1 in central
	found io.delta#delta-storage;4.3.1 in central
	found io.unitycatalog#unitycatalog-client;0.5.0 in central


# Phase 3 — Gold Layer: KPI Aggregations

**What this notebook does:**
- Reads from Silver Delta tables
- Computes 5(+1) business KPIs as aggregated Gold Delta tables
- Each KPI answers a real operational question about the drone fleet

**Gold tables produced:**
| Table | Business Question |
|-------|------------------|
| `gold_delivery_success_rate` | Which drones are most reliable? |
| `gold_avg_delivery_time` | Which routes take longest? |
| `gold_failure_rate_by_cause` | What is causing failures? |
| `gold_battery_efficiency` | Which drones use battery most efficiently? |
| `gold_zone_failure_hotspots` | Which delivery zones have the highest failure rates? |
| `gold_monthly_failure_trend` | Is failure rate improving over time? |

In [3]:
print(f"Reading Silver from : {SILVER_DIR}")
print(f"Writing Gold to     : {GOLD_DIR}")

Reading Silver from : /home/nomaan/projects/Drone_Delivery_Monitoring_and_Failure_Analysis_Pipeline/silver
Writing Gold to     : /home/nomaan/projects/Drone_Delivery_Monitoring_and_Failure_Analysis_Pipeline/gold


In [4]:
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg,
    round as spark_round, min as spark_min,
    max as spark_max, date_format, lit
)

print("Imports loaded")

Imports loaded


## Read Silver Tables

In [5]:
silver_drones = (
    spark.read
    .format("delta")
    .load(str(SILVER_DIR / "silver_drones"))
)

silver_del = (
    spark.read
    .format("delta")
    .load(str(SILVER_DIR / "silver_deliveries"))
)

silver_logs = (
    spark.read
    .format("delta")
    .load(str(SILVER_DIR / "silver_flight_logs"))
)

## KPI 1 — Delivery Success Rate per Drone
**Business question:** Which drones in the fleet are most and least reliable?

Formula: `success_rate_pct = (successful_deliveries / total_deliveries) * 100`

Grouped by `drone_id`, joined with `silver_drones` to include model info.

In [6]:
df_success_rate = (
    silver_del
    .groupBy("drone_id")
    .agg(
        count("*").alias("total_deliveries"),
        spark_sum("failure_flag").alias("total_failures"),
        (count("*") - spark_sum("failure_flag")).alias("successful_deliveries")
    )
    .withColumn(
        "success_rate_pct",
        spark_round(
            (col("successful_deliveries") / col("total_deliveries")) * 100, 2
        )
    )
    .join(silver_drones.select("drone_id", "model", "max_range_km"), "drone_id")
    .orderBy("success_rate_pct")
)

(
    df_success_rate.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(str(GOLD_DIR / "gold_delivery_success_rate"))
)

print(f"gold_delivery_success_rate written — {df_success_rate.count():,} rows")
print("\nBottom 5 drones (least reliable):")
df_success_rate.select("drone_id", "model", "total_deliveries", "total_failures", "success_rate_pct").show(5)
print("Top 5 drones (most reliable):")
df_success_rate.orderBy("success_rate_pct", ascending=False).select("drone_id", "model", "total_deliveries", "total_failures", "success_rate_pct").show(5)

26/07/30 09:03:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

gold_delivery_success_rate written — 100 rows

Bottom 5 drones (least reliable):
+--------+---------+----------------+--------------+----------------+
|drone_id|    model|total_deliveries|total_failures|success_rate_pct|
+--------+---------+----------------+--------------+----------------+
|    D069|  Wing-G2|              63|            20|           68.25|
|    D080|  Wing-G2|              36|            11|           69.44|
|    D005|Skydio-D2|              46|            14|           69.57|
|    D057|  Wing-G2|              60|            18|            70.0|
|    D064| DJI-X500|              52|            15|           71.15|
+--------+---------+----------------+--------------+----------------+
only showing top 5 rows
Top 5 drones (most reliable):
+--------+--------+----------------+--------------+----------------+
|drone_id|   model|total_deliveries|total_failures|success_rate_pct|
+--------+--------+----------------+--------------+----------------+
|    D006|DJI-X500|         

## KPI 2 — Average Delivery Time per Route
**Business question:** Which routes consistently take the longest?

Formula: `avg_delivery_time_mins = AVG(delivery_duration_mins)`

Only successful deliveries included - failed deliveries end early and would skew the average down.

In [7]:
df_avg_delivery_time = (
    silver_del
    .filter(col("failure_flag") == 0)
    .groupBy("source", "destination")
    .agg(
        count("*").alias("total_deliveries"),
        spark_round(avg("delivery_duration_mins"), 2).alias("avg_delivery_time_mins"),
        spark_round(spark_min("delivery_duration_mins"), 2).alias("min_delivery_time_mins"),
        spark_round(spark_max("delivery_duration_mins"), 2).alias("max_delivery_time_mins"),
        spark_round(avg("distance_km"), 2).alias("avg_distance_km")
    )
    .orderBy("avg_delivery_time_mins", ascending=False)
)

(
    df_avg_delivery_time.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(str(GOLD_DIR / "gold_avg_delivery_time"))
)

print(f"gold_avg_delivery_time written — {df_avg_delivery_time.count():,} rows")
print("\nTop 5 slowest routes:")
df_avg_delivery_time.show(5, truncate=False)

gold_avg_delivery_time written — 150 rows

Top 5 slowest routes:
+---------------+-----------+----------------+----------------------+----------------------+----------------------+---------------+
|source         |destination|total_deliveries|avg_delivery_time_mins|min_delivery_time_mins|max_delivery_time_mins|avg_distance_km|
+---------------+-----------+----------------+----------------------+----------------------+----------------------+---------------+
|Hub-Central    |Zone-11    |23              |41.64                 |10.2                  |103.28                |39.48          |
|Warehouse-South|Zone-04    |25              |39.43                 |10.08                 |85.28                 |36.43          |
|Hub-West       |Zone-26    |21              |38.53                 |5.52                  |102.55                |32.78          |
|Warehouse-East |Zone-21    |21              |38.44                 |5.27                  |101.32                |36.63          |
|Hub-Centra

## KPI 3 — Failure Rate by Cause
**Business question:** What is the leading cause of drone failures - battery, signal, or weather?

Formula: `(COUNT of FAILED status/ total_deliveries) * 100 = Failure Rate`

Taking a count of total failed deliveries and then stating the contribution of each known cause of failure.

In [8]:
total_deliveries = silver_del.count()
total_failures   = silver_del.filter(col("failure_flag") == 1).count()

df_failure_by_cause = (
    silver_logs
    .filter(col("failure_flag") == 1)
    .groupBy("failure_cause")
    .agg(count("*").alias("failure_count"))
    .withColumn(
        "pct_of_failures",
        spark_round((col("failure_count") / total_failures) * 100, 2)
    )
    .withColumn(
        "pct_of_total_deliveries",
        spark_round((col("failure_count") / total_deliveries) * 100, 2)
    )
    .orderBy("failure_count", ascending=False)
)

(
    df_failure_by_cause.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(str(GOLD_DIR / "gold_failure_rate_by_cause"))
)

print(f"gold_failure_rate_by_cause written — {df_failure_by_cause.count():,} rows")
print(f"\nTotal deliveries : {total_deliveries:,}")
print(f"Total failures   : {total_failures:,}  ({total_failures/total_deliveries*100:.1f}%)")
print("\nFailure breakdown:")
df_failure_by_cause.show(truncate=False)

gold_failure_rate_by_cause written — 4 rows

Total deliveries : 5,000
Total failures   : 907  (18.1%)

Failure breakdown:
+---------------+-------------+---------------+-----------------------+
|failure_cause  |failure_count|pct_of_failures|pct_of_total_deliveries|
+---------------+-------------+---------------+-----------------------+
|FAILED_BATTERY |335          |36.93          |6.7                    |
|FAILED_SIGNAL  |282          |31.09          |5.64                   |
|UNKNOWN_FAILURE|188          |20.73          |3.76                   |
|FAILED_WEATHER |102          |11.25          |2.04                   |
+---------------+-------------+---------------+-----------------------+



## KPI 4 — Battery Efficiency per Drone
**Business question:** Which drones travel the most distance per unit of battery consumed?

Formula: `battery_efficiency = distance_km / battery_consumed`  
where `battery_consumed = max(battery_level) - min(battery_level)` per delivery.

Using max/min per delivery is valid because battery only decreases during a flight.

In [9]:
# Computing battery consumed per delivery from flight logs
df_battery_consumed = (
    silver_logs
    .groupBy("delivery_id", "drone_id")
    .agg(
        spark_max("battery_level").alias("initial_battery"),
        spark_min("battery_level").alias("final_battery")
    )
    .withColumn(
        "battery_consumed_pct",
        spark_round(col("initial_battery") - col("final_battery"), 2)
    )
    .filter(col("battery_consumed_pct") > 0)
)

# Join with deliveries to get distance_km
df_efficiency_per_delivery = (
    df_battery_consumed
    .join(
        silver_del.select("delivery_id", "distance_km", "failure_flag"),
        "delivery_id"
    )
    .filter(col("failure_flag") == 0)
    .filter(col("distance_km").isNotNull())
    .withColumn(
        "battery_efficiency_km_per_pct",
        spark_round(col("distance_km") / col("battery_consumed_pct"), 3)
    )
)

# Aggregate per drone
df_battery_efficiency = (
    df_efficiency_per_delivery
    .groupBy("drone_id")
    .agg(
        count("*").alias("total_deliveries"),
        spark_round(avg("battery_efficiency_km_per_pct"), 3).alias("avg_km_per_pct_battery"),
        spark_round(avg("battery_consumed_pct"), 2).alias("avg_battery_consumed_pct"),
        spark_round(avg("distance_km"), 2).alias("avg_distance_km")
    )
    .join(silver_drones.select("drone_id", "model"), "drone_id")
    .orderBy("avg_km_per_pct_battery", ascending=False)
)

(
    df_battery_efficiency.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(str(GOLD_DIR / "gold_battery_efficiency"))
)

print(f"gold_battery_efficiency written — {df_battery_efficiency.count():,} rows")
print("\nTop 5 most battery-efficient drones:")
df_battery_efficiency.show(5, truncate=False)

gold_battery_efficiency written — 100 rows

Top 5 most battery-efficient drones:
+--------+----------------+----------------------+------------------------+---------------+----------+
|drone_id|total_deliveries|avg_km_per_pct_battery|avg_battery_consumed_pct|avg_distance_km|model     |
+--------+----------------+----------------------+------------------------+---------------+----------+
|D058    |36              |2.319                 |16.24                   |35.6           |DJI-X500  |
|D086    |35              |2.234                 |12.96                   |28.28          |DJI-X300  |
|D084    |45              |2.219                 |15.49                   |31.23          |Skydio-D2 |
|D011    |47              |2.208                 |10.12                   |20.4           |Wing-G2   |
|D008    |30              |2.191                 |21.05                   |44.85          |Zipline-R1|
+--------+----------------+----------------------+------------------------+---------------+----

## KPI 5 — High-Risk Destination Zones
**Business question:** Which delivery zones have the highest failure rates?

Formula: `failure_rate_pct = (total_failures / total_deliveries) * 100`

Grouped by `destination` across all deliveries.
Zones with consistently high failure rates may indicate:
- GPS dead zones or terrain interference
- Localised weather patterns
- Routes exceeding drone range capability

**Note:** This KPI was explicitly defined in the project document.
High-risk zones directly inform no-fly zone policies and route planning decisions.

In [10]:
# KPI 5 — High-Risk Destination Zones

from pyspark.sql.functions import when

df_zone_hotspots = (
    silver_del
    .groupBy("destination")
    .agg(
        count("*").alias("total_deliveries"),
        spark_sum("failure_flag").alias("total_failures"),
        (count("*") - spark_sum("failure_flag")).alias("successful_deliveries"),
        spark_round(
            spark_sum("failure_flag") / count("*") * 100, 2
        ).alias("failure_rate_pct"),
        spark_round(
            avg("distance_km"), 2
        ).alias("avg_distance_km")
    )
    .withColumn(
        "risk_category",
        when(col("failure_rate_pct") >= 22, "High Risk")
        .when(col("failure_rate_pct") >= 16, "Medium Risk")
        .otherwise("Low Risk")
    )
    .orderBy("failure_rate_pct", ascending=False)
)

(
    df_zone_hotspots.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(str(GOLD_DIR / "gold_zone_failure_hotspots"))
)

print(f"gold_zone_failure_hotspots written — {df_zone_hotspots.count():,} rows")
print("\nAll 30 Destination Zones by Failure Rate:")
df_zone_hotspots.show(30, truncate=False)

gold_zone_failure_hotspots written — 30 rows

All 30 Destination Zones by Failure Rate:
+-----------+----------------+--------------+---------------------+----------------+---------------+-------------+
|destination|total_deliveries|total_failures|successful_deliveries|failure_rate_pct|avg_distance_km|risk_category|
+-----------+----------------+--------------+---------------------+----------------+---------------+-------------+
|Zone-14    |158             |39            |119                  |24.68           |29.37          |High Risk    |
|Zone-18    |147             |35            |112                  |23.81           |32.78          |High Risk    |
|Zone-05    |171             |40            |131                  |23.39           |31.46          |High Risk    |
|Zone-25    |173             |38            |135                  |21.97           |29.0           |Medium Risk  |
|Zone-21    |142             |31            |111                  |21.83           |28.71          |Medium 

In [11]:
print("Risk Category Distribution:")
spark.read.format("delta").load(str(GOLD_DIR / "gold_zone_failure_hotspots")) \
    .groupBy("risk_category") \
    .agg(
        count("*").alias("zone_count"),
        spark_round(avg("failure_rate_pct"), 2).alias("avg_failure_rate_pct"),
        spark_round(avg("total_deliveries"), 0).alias("avg_deliveries_per_zone")
    ) \
    .orderBy("avg_failure_rate_pct", ascending=False) \
    .show(truncate=False)

print("Top 5 Highest Risk Zones:")
spark.read.format("delta").load(str(GOLD_DIR / "gold_zone_failure_hotspots"))\
    .filter(col("risk_category") == "High Risk") \
    .select("destination", "total_deliveries", "total_failures",
            "failure_rate_pct", "avg_distance_km", "risk_category") \
    .show(5, truncate=False)

print("Top 5 Lowest Risk Zones:")
spark.read.format("delta").load(str(GOLD_DIR / "gold_zone_failure_hotspots"))\
    .filter(col("risk_category") == "Low Risk") \
    .select("destination", "total_deliveries", "total_failures",
            "failure_rate_pct", "avg_distance_km", "risk_category") \
    .orderBy("failure_rate_pct") \
    .show(5, truncate=False)

Risk Category Distribution:
+-------------+----------+--------------------+-----------------------+
|risk_category|zone_count|avg_failure_rate_pct|avg_deliveries_per_zone|
+-------------+----------+--------------------+-----------------------+
|High Risk    |3         |23.96               |159.0                  |
|Medium Risk  |19        |18.61               |168.0                  |
|Low Risk     |8         |14.98               |165.0                  |
+-------------+----------+--------------------+-----------------------+

Top 5 Highest Risk Zones:
+-----------+----------------+--------------+----------------+---------------+-------------+
|destination|total_deliveries|total_failures|failure_rate_pct|avg_distance_km|risk_category|
+-----------+----------------+--------------+----------------+---------------+-------------+
|Zone-14    |158             |39            |24.68           |29.37          |High Risk    |
|Zone-18    |147             |35            |23.81           |32.78  

## Additional KPI — Monthly Failure Trend
**Business question:** Is the fleet failure rate improving or worsening month over month?

Covering Oct 2025 — Mar 2026 (6 months of simulated operations).

In [12]:
df_monthly_trend = (
    silver_del
    .withColumn("year_month", date_format(col("start_time"), "yyyy-MM"))
    .groupBy("year_month")
    .agg(
        count("*").alias("total_deliveries"),
        spark_sum("failure_flag").alias("total_failures"),
        (count("*") - spark_sum("failure_flag")).alias("successful_deliveries")
    )
    .withColumn(
        "failure_rate_pct",
        spark_round((col("total_failures") / col("total_deliveries")) * 100, 2)
    )
    .withColumn(
        "success_rate_pct",
        spark_round((col("successful_deliveries") / col("total_deliveries")) * 100, 2)
    )
    .orderBy("year_month")
)

(
    df_monthly_trend.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(str(GOLD_DIR / "gold_monthly_failure_trend"))
)

print(f"gold_monthly_failure_trend written — {df_monthly_trend.count():,} rows")
print("\nMonthly failure trend:")
df_monthly_trend.show(truncate=False)

gold_monthly_failure_trend written — 6 rows

Monthly failure trend:
+----------+----------------+--------------+---------------------+----------------+----------------+
|year_month|total_deliveries|total_failures|successful_deliveries|failure_rate_pct|success_rate_pct|
+----------+----------------+--------------+---------------------+----------------+----------------+
|2025-10   |875             |153           |722                  |17.49           |82.51           |
|2025-11   |802             |143           |659                  |17.83           |82.17           |
|2025-12   |866             |172           |694                  |19.86           |80.14           |
|2026-01   |832             |144           |688                  |17.31           |82.69           |
|2026-02   |791             |137           |654                  |17.32           |82.68           |
|2026-03   |834             |158           |676                  |18.94           |81.06           |
+----------+-----------

## Validation - All Gold Tables

In [13]:
gold_tables = [
    "gold_delivery_success_rate",
    "gold_avg_delivery_time",
    "gold_failure_rate_by_cause",
    "gold_battery_efficiency",
    "gold_zone_failure_hotspots",
    "gold_monthly_failure_trend"
]

print("Gold Layer — Table Summary:")
print(f"  {'Table':<35} {'Rows':>6}")
print(f"  {'-'*42}")
for table in gold_tables:
    rows = (
    spark.read
    .format("delta")
    .load(str(GOLD_DIR / table))
    .count()
)
    print(f"  {table:<35} {rows:>6,}")

print("\nAll 6 Gold KPI tables written successfully")

Gold Layer — Table Summary:
  Table                                 Rows
  ------------------------------------------
  gold_delivery_success_rate             100
  gold_avg_delivery_time                 150
  gold_failure_rate_by_cause               4
  gold_battery_efficiency                100
  gold_zone_failure_hotspots              30
  gold_monthly_failure_trend               6

All 6 Gold KPI tables written successfully


## Business Narrative
This section summarises what the Gold layer reveals about fleet operations.

In [14]:
print("=" * 60)
print("DRONE DELIVERY FLEET — OPERATIONAL SUMMARY")
print("=" * 60)

# Overall fleet health
total = silver_del.count()
failed = silver_del.filter(col("failure_flag") == 1).count()

print("\nFleet Overview")
print(f"  Total deliveries     : {total:,}")
print(f"  Successful           : {total - failed:,}  ({(total - failed) / total * 100:.1f}%)")
print(f"  Failed               : {failed:,}  ({failed / total * 100:.1f}%)")

# Worst performing drone
worst = (
    spark.read
    .format("delta")
    .load(str(GOLD_DIR / "gold_delivery_success_rate"))
    .first()
)

print(
    f"\nLowest reliability drone : "
    f"{worst['drone_id']} ({worst['model']}) — "
    f"{worst['success_rate_pct']}% success rate"
)

# Top failure cause
top_cause = (
    spark.read
    .format("delta")
    .load(str(GOLD_DIR / "gold_failure_rate_by_cause"))
    .first()
)

print(
    f"Leading failure cause    : "
    f"{top_cause['failure_cause']} "
    f"({top_cause['pct_of_failures']}% of all failures)"
)

# Best battery efficiency
best_batt = (
    spark.read
    .format("delta")
    .load(str(GOLD_DIR / "gold_battery_efficiency"))
    .first()
)

print(
    f"Most efficient drone     : "
    f"{best_batt['drone_id']} ({best_batt['model']}) — "
    f"{best_batt['avg_km_per_pct_battery']} km per 1% battery"
)

# Highest risk zone
worst_zone = (
    spark.read
    .format("delta")
    .load(str(GOLD_DIR / "gold_zone_failure_hotspots"))
    .first()
)

print(
    f"Highest risk zone        : "
    f"{worst_zone['destination']} — "
    f"{worst_zone['failure_rate_pct']}% failure rate "
    f"({worst_zone['risk_category']})"
)

print("\n" + "=" * 60)

DRONE DELIVERY FLEET — OPERATIONAL SUMMARY

Fleet Overview
  Total deliveries     : 5,000
  Successful           : 4,093  (81.9%)
  Failed               : 907  (18.1%)

Lowest reliability drone : D069 (Wing-G2) — 68.25% success rate
Leading failure cause    : FAILED_BATTERY (36.93% of all failures)
Most efficient drone     : D058 (DJI-X500) — 2.319 km per 1% battery
Highest risk zone        : Zone-14 — 24.68% failure rate (High Risk)

